# تدريب نواة على Colab (GPU)
قبل البدء: القائمة **Runtime ← Change runtime type ← T4 GPU**.
ثم شغّل الخلايا بالترتيب. التدريب من الصفر، بدون أي نموذج جاهز.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "لا يوجد! فعّل GPU من Runtime")

**1) ارفع `Nuwaa_updated.zip`** (وأي ملفات بيانات إضافية تحطها في `data/` بصيغة jsonl: كل سطر `{"user": ..., "bot": ...}`).

In [ ]:
from google.colab import files
up = files.upload()
name = next(iter(up))
!rm -rf Nuwaa && unzip -q "$name" && ls Nuwaa
%cd Nuwaa
!python tools/gen_knowledge.py
!python -m unittest discover -s tests 2>&1 | tail -3

**2) اختر المرحلة والخطوات.**
- ابدأ بـ `s5m` ثم `s10m`. لا تقفز لمرحلة كبيرة قبل ما تتحسن الدرجة.
- `STEPS` الأكبر = وقت أطول. جرّب 5000 أولا لتشوف الوقت.

In [ ]:
STAGE = "s5m"      # s2m, s5m, s10m, s20m, s30m, s45m, s80m
STEPS = 5000
BS = 32

**3) التدريب وكل الفحوصات** (يطبع الخسارة كل 100 خطوة، وفي النهاية تقرير كامل).

In [ ]:
!python -m nuwaa.core.stage_cli set $STAGE
!python -m nuwaa.core.run_stage --stage $STAGE --steps $STEPS --bs $BS

**4) اقرأ النتيجة بصدق.**
- `benchmark.total` هي الدرجة على أسئلة ما دخلت التدريب. هذا الرقم المهم، مو الخسارة.
- أقل من 0.5 = النموذج ما صار صالحا، لا تحطه في التطبيق.
- `all_ok` لازم تكون True.

In [ ]:
import json
r = json.load(open(f"stage_reports/{STAGE}.json"))
print("all_ok:", r["all_ok"])
print("الفحوصات:", r["checks"])
print("الدرجة:", r["metrics"]["benchmark"])
print("الخسارة:", r["metrics"]["training"])
good = r["all_ok"] and r["metrics"]["benchmark"]["total"] >= 0.5
print("\nجاهز للتطبيق؟", "نعم" if good else "لا. زد البيانات أو الخطوات ثم أعد التدريب")

**5) نزّل النموذج** وضعه في `android/app/src/main/assets/model.bin` ثم ابنِ التطبيق (أو ارفعه لمستودع GitHub في `model_accepted/model.bin`).
> لا تنزّله إلا إذا قالت الخلية السابقة «نعم».

In [ ]:
if good:
    files.download(f"stage_artifacts/{STAGE}/model.bin")
else:
    print("النموذج لم يجتز البوابة، لن أنزّله.")